In [1]:
# TEJAS-EV Impact Analysis
# Step 1: Load and inspect the existing raw dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_path = "../data/raw/TEJAS_RAW_DATA.csv"

impact_df = pd.read_csv(raw_path)

print("TEJAS-EV impact analysis dataset loaded successfully.")
print("Shape:", impact_df.shape)

print("\nRelevant columns:")
impact_columns = [
    "Depot ID",
    "Depot Name",
    "District",
    "Month",
    "Year",
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Estimated Diesel Litres",
    "Estimated CO2 (Tonnes)",
    "Diesel Cost (INR/KM)",
    "EV Cost (INR/KM)",
    "Estimated EV Energy (MWh)",
    "Potential EV OPEX Saving (INR)",
    "Electricity Tariff (INR/kWh)",
    "EV Battery Capacity (kWh)",
    "EV Range (KM)",
    "Charging Power (kW)"
]

print(impact_df[impact_columns].head().to_string(index=False))

print("\nMissing values in relevant columns:")
print(impact_df[impact_columns].isnull().sum())

TEJAS-EV impact analysis dataset loaded successfully.
Shape: (5520, 20)

Relevant columns:
 Depot ID Depot Name       District   Month  Year  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Estimated Diesel Litres  Estimated CO2 (Tonnes)  Diesel Cost (INR/KM)  EV Cost (INR/KM)  Estimated EV Energy (MWh)  Potential EV OPEX Saving (INR)  Electricity Tariff (INR/kWh)  EV Battery Capacity (kWh)  EV Range (KM)  Charging Power (kW)
KSRTC-001      ADOOR Pathanamthitta 2021-04  2021               30                   24        129270      163058             31679.941184               84.902242                    51                27                        162                         3102480                             8                        250            200                  120
KSRTC-001      ADOOR Pathanamthitta 2021-05  2021               30                   24        129616      167645             31764.734714               85.129489                    51               

In [2]:
# Step 2: Validate Potential EV OPEX Saving

impact_df["Calculated OPEX Saving (INR)"] = (
    impact_df["Diesel Cost (INR/KM)"]
    - impact_df["EV Cost (INR/KM)"]
) * impact_df["Effective KM"]

impact_df["OPEX Difference (INR)"] = (
    impact_df["Potential EV OPEX Saving (INR)"]
    - impact_df["Calculated OPEX Saving (INR)"]
)

print("OPEX calculation validation")
print("-" * 50)

print(
    "Maximum absolute difference:",
    impact_df["OPEX Difference (INR)"].abs().max()
)

print(
    "Number of mismatches:",
    (impact_df["OPEX Difference (INR)"].abs() > 0.01).sum()
)

print("\nSample validation:")
print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Diesel Cost (INR/KM)",
            "EV Cost (INR/KM)",
            "Potential EV OPEX Saving (INR)",
            "Calculated OPEX Saving (INR)",
            "OPEX Difference (INR)"
        ]
    ].head(10).to_string(index=False)
)

OPEX calculation validation
--------------------------------------------------
Maximum absolute difference: 0
Number of mismatches: 0

Sample validation:
 Depot ID  Effective KM  Diesel Cost (INR/KM)  EV Cost (INR/KM)  Potential EV OPEX Saving (INR)  Calculated OPEX Saving (INR)  OPEX Difference (INR)
KSRTC-001        129270                    51                27                         3102480                       3102480                      0
KSRTC-001        129616                    51                27                         3110784                       3110784                      0
KSRTC-001        132462                    51                27                         3179088                       3179088                      0
KSRTC-001        140413                    51                27                         3369912                       3369912                      0
KSRTC-001        126289                    51                27                         3030936      

In [3]:
# Step 3: Validate Estimated Diesel CO2

# Calculate the implied CO2 emission factor from the existing dataset
impact_df["Calculated CO2 (Tonnes)"] = (
    impact_df["Estimated Diesel Litres"]
    * impact_df["Estimated CO2 (Tonnes)"]
    / impact_df["Estimated Diesel Litres"]
)

impact_df["CO2 Difference (Tonnes)"] = (
    impact_df["Estimated CO2 (Tonnes)"]
    - impact_df["Calculated CO2 (Tonnes)"]
)

print("CO2 data inspection")
print("-" * 50)

print(
    "Average CO2 per litre of diesel (tonnes/litre):",
    (
        impact_df["Estimated CO2 (Tonnes)"]
        / impact_df["Estimated Diesel Litres"]
    ).mean()
)

print(
    "Minimum CO2 per litre:",
    (
        impact_df["Estimated CO2 (Tonnes)"]
        / impact_df["Estimated Diesel Litres"]
    ).min()
)

print(
    "Maximum CO2 per litre:",
    (
        impact_df["Estimated CO2 (Tonnes)"]
        / impact_df["Estimated Diesel Litres"]
    ).max()
)

print("\nSample CO2 values:")
print(
    impact_df[
        [
            "Depot ID",
            "Estimated Diesel Litres",
            "Estimated CO2 (Tonnes)"
        ]
    ].head(10).to_string(index=False)
)

CO2 data inspection
--------------------------------------------------
Average CO2 per litre of diesel (tonnes/litre): 0.00268
Minimum CO2 per litre: 0.0026799999999999992
Maximum CO2 per litre: 0.0026800000000000014

Sample CO2 values:
 Depot ID  Estimated Diesel Litres  Estimated CO2 (Tonnes)
KSRTC-001             31679.941184               84.902242
KSRTC-001             31764.734714               85.129489
KSRTC-001             32462.198260               86.998691
KSRTC-001             34410.733979               92.220767
KSRTC-001             30949.393457               82.944374
KSRTC-001             29467.957358               78.974126
KSRTC-001             31277.784585               83.824463
KSRTC-001             31601.274354               84.691415
KSRTC-001             30510.966793               81.769391
KSRTC-001             31770.126210               85.143938


In [4]:
# Step 4: Validate CO2 calculation using the dataset emission factor

DIESEL_CO2_FACTOR = 0.00268  # tonnes CO2 per litre of diesel

impact_df["Calculated CO2 (Tonnes)"] = (
    impact_df["Estimated Diesel Litres"]
    * DIESEL_CO2_FACTOR
)

impact_df["CO2 Difference (Tonnes)"] = (
    impact_df["Estimated CO2 (Tonnes)"]
    - impact_df["Calculated CO2 (Tonnes)"]
)

print("CO2 calculation validation")
print("-" * 50)

print(
    "Maximum absolute difference:",
    impact_df["CO2 Difference (Tonnes)"].abs().max()
)

print(
    "Number of mismatches:",
    (impact_df["CO2 Difference (Tonnes)"].abs() > 1e-9).sum()
)

print("\nSample validation:")

print(
    impact_df[
        [
            "Depot ID",
            "Estimated Diesel Litres",
            "Estimated CO2 (Tonnes)",
            "Calculated CO2 (Tonnes)",
            "CO2 Difference (Tonnes)"
        ]
    ].head(10).to_string(index=False)
)

CO2 calculation validation
--------------------------------------------------
Maximum absolute difference: 2.2737367544323206e-13
Number of mismatches: 0

Sample validation:
 Depot ID  Estimated Diesel Litres  Estimated CO2 (Tonnes)  Calculated CO2 (Tonnes)  CO2 Difference (Tonnes)
KSRTC-001             31679.941184               84.902242                84.902242             0.000000e+00
KSRTC-001             31764.734714               85.129489                85.129489            -1.421085e-14
KSRTC-001             32462.198260               86.998691                86.998691             0.000000e+00
KSRTC-001             34410.733979               92.220767                92.220767             1.421085e-14
KSRTC-001             30949.393457               82.944374                82.944374            -1.421085e-14
KSRTC-001             29467.957358               78.974126                78.974126             1.421085e-14
KSRTC-001             31277.784585               83.824463     

In [5]:
# Step 5: Validate diesel fuel consumption efficiency

impact_df["Diesel Efficiency (KM/L)"] = (
    impact_df["Effective KM"]
    / impact_df["Estimated Diesel Litres"]
)

print("Diesel fuel consumption validation")
print("-" * 50)

print(
    "Average diesel efficiency (KM/L):",
    impact_df["Diesel Efficiency (KM/L)"].mean()
)

print(
    "Minimum diesel efficiency (KM/L):",
    impact_df["Diesel Efficiency (KM/L)"].min()
)

print(
    "Maximum diesel efficiency (KM/L):",
    impact_df["Diesel Efficiency (KM/L)"].max()
)

print("\nPercentiles:")
print(
    impact_df["Diesel Efficiency (KM/L)"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
)

print("\nSample records:")
print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Estimated Diesel Litres",
            "Diesel Efficiency (KM/L)"
        ]
    ].head(10).to_string(index=False)
)

Diesel fuel consumption validation
--------------------------------------------------
Average diesel efficiency (KM/L): 4.0805
Minimum diesel efficiency (KM/L): 4.080499999999999
Maximum diesel efficiency (KM/L): 4.080500000000001

Percentiles:
count    5.520000e+03
mean     4.080500e+00
std      2.808921e-16
min      4.080500e+00
1%       4.080500e+00
5%       4.080500e+00
25%      4.080500e+00
50%      4.080500e+00
75%      4.080500e+00
95%      4.080500e+00
99%      4.080500e+00
max      4.080500e+00
Name: Diesel Efficiency (KM/L), dtype: float64

Sample records:
 Depot ID  Effective KM  Estimated Diesel Litres  Diesel Efficiency (KM/L)
KSRTC-001        129270             31679.941184                    4.0805
KSRTC-001        129616             31764.734714                    4.0805
KSRTC-001        132462             32462.198260                    4.0805
KSRTC-001        140413             34410.733979                    4.0805
KSRTC-001        126289             30949.393457    

In [6]:
# Step 6: Validate the diesel consumption formula

DIESEL_EFFICIENCY_KM_PER_L = 4.0805

impact_df["Calculated Diesel Litres"] = (
    impact_df["Effective KM"]
    / DIESEL_EFFICIENCY_KM_PER_L
)

impact_df["Diesel Litres Difference"] = (
    impact_df["Estimated Diesel Litres"]
    - impact_df["Calculated Diesel Litres"]
)

print("Diesel consumption formula validation")
print("-" * 50)

print(
    "Maximum absolute difference:",
    impact_df["Diesel Litres Difference"].abs().max()
)

print(
    "Number of mismatches:",
    (
        impact_df["Diesel Litres Difference"].abs() > 1e-9
    ).sum()
)

print("\nFormula used:")
print(
    "Estimated Diesel Litres = Effective KM /",
    DIESEL_EFFICIENCY_KM_PER_L
)

Diesel consumption formula validation
--------------------------------------------------
Maximum absolute difference: 5.820766091346741e-11
Number of mismatches: 0

Formula used:
Estimated Diesel Litres = Effective KM / 4.0805


In [7]:
# Step 7: Inspect EV energy consumption

impact_df["EV Energy per KM (MWh/KM)"] = (
    impact_df["Estimated EV Energy (MWh)"]
    / impact_df["Effective KM"]
)

print("EV energy consumption validation")
print("-" * 50)

print(
    "Average EV energy per KM (MWh/KM):",
    impact_df["EV Energy per KM (MWh/KM)"].mean()
)

print(
    "Minimum EV energy per KM:",
    impact_df["EV Energy per KM (MWh/KM)"].min()
)

print(
    "Maximum EV energy per KM:",
    impact_df["EV Energy per KM (MWh/KM)"].max()
)

print("\nEV energy consumption in kWh/km:")

print(
    "Average:",
    impact_df["EV Energy per KM (MWh/KM)"].mean() * 1000
)

print("\nSample records:")

print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Estimated EV Energy (MWh)",
            "EV Energy per KM (MWh/KM)"
        ]
    ].head(10).to_string(index=False)
)

EV energy consumption validation
--------------------------------------------------
Average EV energy per KM (MWh/KM): 0.0012500739039349449
Minimum EV energy per KM: 0.0012463710841908468
Maximum EV energy per KM: 0.001254138657569981

EV energy consumption in kWh/km:
Average: 1.2500739039349449

Sample records:
 Depot ID  Effective KM  Estimated EV Energy (MWh)  EV Energy per KM (MWh/KM)
KSRTC-001        129270                        162                   0.001253
KSRTC-001        129616                        162                   0.001250
KSRTC-001        132462                        166                   0.001253
KSRTC-001        140413                        176                   0.001253
KSRTC-001        126289                        158                   0.001251
KSRTC-001        120244                        150                   0.001247
KSRTC-001        127629                        160                   0.001254
KSRTC-001        128949                        161           

In [8]:
# Step 8: Validate the EV energy calculation

EV_ENERGY_KWH_PER_KM = 1.25

impact_df["Calculated EV Energy (MWh)"] = (
    impact_df["Effective KM"]
    * EV_ENERGY_KWH_PER_KM
    / 1000
)

impact_df["EV Energy Difference (MWh)"] = (
    impact_df["Estimated EV Energy (MWh)"]
    - impact_df["Calculated EV Energy (MWh)"]
)

print("EV energy calculation validation")
print("-" * 50)

print(
    "Mean absolute difference (MWh):",
    impact_df["EV Energy Difference (MWh)"].abs().mean()
)

print(
    "Maximum absolute difference (MWh):",
    impact_df["EV Energy Difference (MWh)"].abs().max()
)

print("\nSample validation:")

print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Estimated EV Energy (MWh)",
            "Calculated EV Energy (MWh)",
            "EV Energy Difference (MWh)"
        ]
    ].head(10).to_string(index=False)
)

EV energy calculation validation
--------------------------------------------------
Mean absolute difference (MWh): 0.24767663043478297
Maximum absolute difference (MWh): 0.5487500000000409

Sample validation:
 Depot ID  Effective KM  Estimated EV Energy (MWh)  Calculated EV Energy (MWh)  EV Energy Difference (MWh)
KSRTC-001        129270                        162                   161.58750                     0.41250
KSRTC-001        129616                        162                   162.02000                    -0.02000
KSRTC-001        132462                        166                   165.57750                     0.42250
KSRTC-001        140413                        176                   175.51625                     0.48375
KSRTC-001        126289                        158                   157.86125                     0.13875
KSRTC-001        120244                        150                   150.30500                    -0.30500
KSRTC-001        127629                  

In [9]:
# Step 9: Check whether EV energy values are rounded to whole MWh

impact_df["Rounded Calculated EV Energy (MWh)"] = (
    impact_df["Calculated EV Energy (MWh)"].round()
)

impact_df["EV Energy Rounding Difference"] = (
    impact_df["Estimated EV Energy (MWh)"]
    - impact_df["Rounded Calculated EV Energy (MWh)"]
)

print("EV energy rounding validation")
print("-" * 50)

print(
    "Maximum absolute difference after rounding:",
    impact_df["EV Energy Rounding Difference"].abs().max()
)

print(
    "Number of mismatches after rounding:",
    (
        impact_df["EV Energy Rounding Difference"].abs() > 1e-9
    ).sum()
)

print("\nUnique differences after rounding:")

print(
    impact_df["EV Energy Rounding Difference"]
    .round(6)
    .value_counts()
    .head(10)
)

EV energy rounding validation
--------------------------------------------------
Maximum absolute difference after rounding: 1.0
Number of mismatches after rounding: 145

Unique differences after rounding:
EV Energy Rounding Difference
0.0    5375
1.0     145
Name: count, dtype: int64


In [10]:
# Step 9A: Inspect EV energy exceptions

exceptions = impact_df[
    impact_df["EV Energy Rounding Difference"].abs() > 1e-9
].copy()

print("Number of EV energy exceptions:", len(exceptions))

print("\nException records:")
print(
    exceptions[
        [
            "Depot ID",
            "Effective KM",
            "Estimated EV Energy (MWh)",
            "Calculated EV Energy (MWh)",
            "EV Energy Rounding Difference"
        ]
    ].head(20).to_string(index=False)
)

print("\nException difference counts:")
print(
    exceptions["EV Energy Rounding Difference"]
    .value_counts()
)

Number of EV energy exceptions: 145

Exception records:
 Depot ID  Effective KM  Estimated EV Energy (MWh)  Calculated EV Energy (MWh)  EV Energy Rounding Difference
KSRTC-001        226790                        284                   283.48750                            1.0
KSRTC-002        485993                        608                   607.49125                            1.0
KSRTC-003        361985                        453                   452.48125                            1.0
KSRTC-004        388384                        486                   485.48000                            1.0
KSRTC-004        349181                        437                   436.47625                            1.0
KSRTC-004        337981                        423                   422.47625                            1.0
KSRTC-005        325181                        407                   406.47625                            1.0
KSRTC-006        302791                        379              

In [11]:
# Step 10: Validate EV electricity cost methodology

impact_df["Calculated EV Energy Cost (INR)"] = (
    impact_df["Estimated EV Energy (MWh)"]
    * 1000
    * impact_df["Electricity Tariff (INR/kWh)"]
)

print("EV electricity cost validation")
print("-" * 50)

print(
    impact_df[
        [
            "Depot ID",
            "Estimated EV Energy (MWh)",
            "Electricity Tariff (INR/kWh)",
            "Calculated EV Energy Cost (INR)"
        ]
    ].head(10).to_string(index=False)
)

EV electricity cost validation
--------------------------------------------------
 Depot ID  Estimated EV Energy (MWh)  Electricity Tariff (INR/kWh)  Calculated EV Energy Cost (INR)
KSRTC-001                        162                             8                          1296000
KSRTC-001                        162                             8                          1296000
KSRTC-001                        166                             8                          1328000
KSRTC-001                        176                             8                          1408000
KSRTC-001                        158                             8                          1264000
KSRTC-001                        150                             8                          1200000
KSRTC-001                        160                             8                          1280000
KSRTC-001                        161                             8                          1288000
KSRTC-001         

In [12]:
# Step 11: Validate complete diesel-vs-EV operating cost relationship

impact_df["Calculated Diesel Cost (INR)"] = (
    impact_df["Effective KM"]
    * impact_df["Diesel Cost (INR/KM)"]
)

impact_df["Calculated EV Cost (INR)"] = (
    impact_df["Estimated EV Energy (MWh)"]
    * 1000
    * impact_df["Electricity Tariff (INR/kWh)"]
)

impact_df["Calculated OPEX Saving from Energy (INR)"] = (
    impact_df["Calculated Diesel Cost (INR)"]
    - impact_df["Calculated EV Cost (INR)"]
)

impact_df["OPEX Energy Difference (INR)"] = (
    impact_df["Potential EV OPEX Saving (INR)"]
    - impact_df["Calculated OPEX Saving from Energy (INR)"]
)

print("Complete EV vs Diesel OPEX validation")
print("-" * 50)

print("Average diesel operating cost (INR):",
      impact_df["Calculated Diesel Cost (INR)"].mean())

print("Average EV electricity cost (INR):",
      impact_df["Calculated EV Cost (INR)"].mean())

print("Average dataset OPEX saving (INR):",
      impact_df["Potential EV OPEX Saving (INR)"].mean())

print("\nDifference caused by using rounded EV energy:")

print(
    "Mean absolute difference (INR):",
    impact_df["OPEX Energy Difference (INR)"].abs().mean()
)

print(
    "Maximum absolute difference (INR):",
    impact_df["OPEX Energy Difference (INR)"].abs().max()
)

print("\nSample validation:")

print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Estimated EV Energy (MWh)",
            "Diesel Cost (INR/KM)",
            "Electricity Tariff (INR/kWh)",
            "Potential EV OPEX Saving (INR)",
            "Calculated OPEX Saving from Energy (INR)",
            "OPEX Energy Difference (INR)"
        ]
    ].head(10).to_string(index=False)
)

Complete EV vs Diesel OPEX validation
--------------------------------------------------
Average diesel operating cost (INR): 17913717.810869563
Average EV electricity cost (INR): 3512682.6086956523
Average dataset OPEX saving (INR): 8429984.852173913

Difference caused by using rounded EV energy:
Mean absolute difference (INR): 5971050.35
Maximum absolute difference (INR): 13958938

Sample validation:
 Depot ID  Effective KM  Estimated EV Energy (MWh)  Diesel Cost (INR/KM)  Electricity Tariff (INR/kWh)  Potential EV OPEX Saving (INR)  Calculated OPEX Saving from Energy (INR)  OPEX Energy Difference (INR)
KSRTC-001        129270                        162                    51                             8                         3102480                                   5296770                      -2194290
KSRTC-001        129616                        162                    51                             8                         3110784                                   5314416    

In [13]:
# Step 12: Inspect EV cost assumptions

print("EV cost assumption inspection")
print("-" * 50)

print("Unique EV Cost (INR/KM):")
print(
    impact_df["EV Cost (INR/KM)"]
    .value_counts()
    .sort_index()
)

print("\nUnique Diesel Cost (INR/KM):")
print(
    impact_df["Diesel Cost (INR/KM)"]
    .value_counts()
    .sort_index()
)

print("\nUnique Electricity Tariff (INR/kWh):")
print(
    impact_df["Electricity Tariff (INR/kWh)"]
    .value_counts()
    .sort_index()
)

print("\nEV Battery Capacity (kWh):")
print(
    impact_df["EV Battery Capacity (kWh)"]
    .value_counts()
    .sort_index()
)

print("\nEV Range (KM):")
print(
    impact_df["EV Range (KM)"]
    .value_counts()
    .sort_index()
)

print("\nCharging Power (kW):")
print(
    impact_df["Charging Power (kW)"]
    .value_counts()
    .sort_index()
)

EV cost assumption inspection
--------------------------------------------------
Unique EV Cost (INR/KM):
EV Cost (INR/KM)
27    5520
Name: count, dtype: int64

Unique Diesel Cost (INR/KM):
Diesel Cost (INR/KM)
51    5520
Name: count, dtype: int64

Unique Electricity Tariff (INR/kWh):
Electricity Tariff (INR/kWh)
8    5520
Name: count, dtype: int64

EV Battery Capacity (kWh):
EV Battery Capacity (kWh)
250    5520
Name: count, dtype: int64

EV Range (KM):
EV Range (KM)
200    5520
Name: count, dtype: int64

Charging Power (kW):
Charging Power (kW)
120    5520
Name: count, dtype: int64


In [14]:
# Step 13: Compare EV operating-cost assumption with
# energy-based electricity cost

EV_COST_PER_KM = 27
EV_ENERGY_KWH_PER_KM = 1.25
ELECTRICITY_TARIFF = 8

energy_based_ev_cost_per_km = (
    EV_ENERGY_KWH_PER_KM * ELECTRICITY_TARIFF
)

print("EV cost methodology comparison")
print("-" * 50)

print("Dataset EV operating cost:",
      EV_COST_PER_KM, "INR/km")

print("Energy-based electricity cost:",
      energy_based_ev_cost_per_km, "INR/km")

print(
    "Difference:",
    EV_COST_PER_KM - energy_based_ev_cost_per_km,
    "INR/km"
)

print(
    "Dataset EV cost is",
    EV_COST_PER_KM / energy_based_ev_cost_per_km,
    "times the energy-only electricity cost."
)

EV cost methodology comparison
--------------------------------------------------
Dataset EV operating cost: 27 INR/km
Energy-based electricity cost: 10.0 INR/km
Difference: 17.0 INR/km
Dataset EV cost is 2.7 times the energy-only electricity cost.


In [15]:
# Step 14: Compare annual EV savings under the two cost methodologies

impact_df["Diesel Cost (INR)"] = (
    impact_df["Effective KM"]
    * impact_df["Diesel Cost (INR/KM)"]
)

impact_df["EV Cost - Dataset Assumption (INR)"] = (
    impact_df["Effective KM"]
    * EV_COST_PER_KM
)

impact_df["EV Electricity Cost (INR)"] = (
    impact_df["Effective KM"]
    * EV_ENERGY_KWH_PER_KM
    * impact_df["Electricity Tariff (INR/kWh)"]
)

impact_df["Saving - Dataset Cost Model (INR)"] = (
    impact_df["Diesel Cost (INR)"]
    - impact_df["EV Cost - Dataset Assumption (INR)"]
)

impact_df["Saving - Energy Cost Model (INR)"] = (
    impact_df["Diesel Cost (INR)"]
    - impact_df["EV Electricity Cost (INR)"]
)

print("EV vs Diesel savings comparison")
print("-" * 50)

print(
    "Average diesel cost (INR):",
    impact_df["Diesel Cost (INR)"].mean()
)

print(
    "Average EV cost - dataset model (INR):",
    impact_df["EV Cost - Dataset Assumption (INR)"].mean()
)

print(
    "Average EV electricity cost (INR):",
    impact_df["EV Electricity Cost (INR)"].mean()
)

print()

print(
    "Average saving - dataset cost model (INR):",
    impact_df["Saving - Dataset Cost Model (INR)"].mean()
)

print(
    "Average saving - energy cost model (INR):",
    impact_df["Saving - Energy Cost Model (INR)"].mean()
)

print()

print("Sample comparison:")

print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Diesel Cost (INR)",
            "EV Cost - Dataset Assumption (INR)",
            "EV Electricity Cost (INR)",
            "Saving - Dataset Cost Model (INR)",
            "Saving - Energy Cost Model (INR)"
        ]
    ].head(10).to_string(index=False)
)

EV vs Diesel savings comparison
--------------------------------------------------
Average diesel cost (INR): 17913717.810869563
Average EV cost - dataset model (INR): 9483732.958695652
Average EV electricity cost (INR): 3512493.688405797

Average saving - dataset cost model (INR): 8429984.852173913
Average saving - energy cost model (INR): 14401224.122463768

Sample comparison:
 Depot ID  Effective KM  Diesel Cost (INR)  EV Cost - Dataset Assumption (INR)  EV Electricity Cost (INR)  Saving - Dataset Cost Model (INR)  Saving - Energy Cost Model (INR)
KSRTC-001        129270            6592770                             3490290                  1292700.0                            3102480                         5300070.0
KSRTC-001        129616            6610416                             3499632                  1296160.0                            3110784                         5314256.0
KSRTC-001        132462            6755562                             3576474               

In [16]:
# Step 15: Environmental impact of EV transition

impact_df["CO2 Reduction (Tonnes)"] = (
    impact_df["Estimated CO2 (Tonnes)"]
)

impact_df["CO2 Reduction (%)"] = 100.0

print("Environmental impact of EV transition")
print("-" * 50)

print(
    "Average diesel CO2 emissions (Tonnes):",
    impact_df["Estimated CO2 (Tonnes)"].mean()
)

print(
    "Average potential CO2 reduction (Tonnes):",
    impact_df["CO2 Reduction (Tonnes)"].mean()
)

print(
    "Total diesel CO2 emissions (Tonnes):",
    impact_df["Estimated CO2 (Tonnes)"].sum()
)

print(
    "Total potential CO2 reduction (Tonnes):",
    impact_df["CO2 Reduction (Tonnes)"].sum()
)

print(
    "Assumed CO2 reduction under full EV transition:",
    impact_df["CO2 Reduction (%)"].iloc[0],
    "%"
)

print()

print("Sample:")
print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Estimated Diesel Litres",
            "Estimated CO2 (Tonnes)",
            "CO2 Reduction (Tonnes)",
            "CO2 Reduction (%)"
        ]
    ].head(10).to_string(index=False)
)

Environmental impact of EV transition
--------------------------------------------------
Average diesel CO2 emissions (Tonnes): 230.69435326375532
Average potential CO2 reduction (Tonnes): 230.69435326375532
Total diesel CO2 emissions (Tonnes): 1273432.8300159294
Total potential CO2 reduction (Tonnes): 1273432.8300159294
Assumed CO2 reduction under full EV transition: 100.0 %

Sample:
 Depot ID  Effective KM  Estimated Diesel Litres  Estimated CO2 (Tonnes)  CO2 Reduction (Tonnes)  CO2 Reduction (%)
KSRTC-001        129270             31679.941184               84.902242               84.902242              100.0
KSRTC-001        129616             31764.734714               85.129489               85.129489              100.0
KSRTC-001        132462             32462.198260               86.998691               86.998691              100.0
KSRTC-001        140413             34410.733979               92.220767               92.220767              100.0
KSRTC-001        126289         

In [17]:
# Step 16: Annual environmental impact

annual_environment = (
    impact_df
    .groupby("Year")
    .agg(
        Total_Effective_KM=("Effective KM", "sum"),
        Total_Diesel_Litres=("Estimated Diesel Litres", "sum"),
        Total_CO2_Tonnes=("Estimated CO2 (Tonnes)", "sum"),
        Potential_CO2_Reduction_Tonnes=("CO2 Reduction (Tonnes)", "sum")
    )
    .reset_index()
)

annual_environment["CO2_Reduction_Percent"] = (
    annual_environment["Potential_CO2_Reduction_Tonnes"]
    / annual_environment["Total_CO2_Tonnes"]
    * 100
)

print("Annual environmental impact")
print("-" * 70)

print(
    annual_environment.to_string(index=False)
)

print()
print(
    "Total potential CO2 reduction across all records:",
    round(
        annual_environment["Potential_CO2_Reduction_Tonnes"].sum(),
        2
    ),
    "tonnes"
)

Annual environmental impact
----------------------------------------------------------------------
 Year  Total_Effective_KM  Total_Diesel_Litres  Total_CO2_Tonnes  Potential_CO2_Reduction_Tonnes  CO2_Reduction_Percent
 2021           217354868         5.326672e+07     142754.820792                   142754.820792                  100.0
 2022           391403336         9.592044e+07     257066.766445                   257066.766445                  100.0
 2023           450041858         1.102909e+08     295579.507276                   295579.507276                  100.0
 2024           401406703         9.837194e+07     263636.800402                   263636.800402                  100.0
 2025           382855996         9.382576e+07     251453.025188                   251453.025188                  100.0
 2026            95833755         2.348579e+07      62941.909913                    62941.909913                  100.0

Total potential CO2 reduction across all records: 1273432.83

In [18]:
# Step 17: EV energy requirement analysis

EV_ENERGY_KWH_PER_KM = 1.25

impact_df["Calculated EV Energy (MWh)"] = (
    impact_df["Effective KM"]
    * EV_ENERGY_KWH_PER_KM
    / 1000
)

impact_df["EV Electricity Cost (INR)"] = (
    impact_df["Calculated EV Energy (MWh)"]
    * 1000
    * impact_df["Electricity Tariff (INR/kWh)"]
)

print("EV energy requirement analysis")
print("-" * 60)

print(
    "Assumed EV energy consumption:",
    EV_ENERGY_KWH_PER_KM,
    "kWh/km"
)

print(
    "Average EV energy requirement (MWh):",
    impact_df["Calculated EV Energy (MWh)"].mean()
)

print(
    "Total EV energy requirement (MWh):",
    impact_df["Calculated EV Energy (MWh)"].sum()
)

print(
    "Total EV energy requirement (GWh):",
    impact_df["Calculated EV Energy (MWh)"].sum() / 1000
)

print(
    "Average EV electricity cost (INR):",
    impact_df["EV Electricity Cost (INR)"].mean()
)

print(
    "Total EV electricity cost (INR):",
    impact_df["EV Electricity Cost (INR)"].sum()
)

print()

print("Sample:")
print(
    impact_df[
        [
            "Depot ID",
            "Effective KM",
            "Calculated EV Energy (MWh)",
            "Electricity Tariff (INR/kWh)",
            "EV Electricity Cost (INR)"
        ]
    ].head(10).to_string(index=False)
)

EV energy requirement analysis
------------------------------------------------------------
Assumed EV energy consumption: 1.25 kWh/km
Average EV energy requirement (MWh): 439.0617110507246
Total EV energy requirement (MWh): 2423620.645
Total EV energy requirement (GWh): 2423.620645
Average EV electricity cost (INR): 3512493.688405797
Total EV electricity cost (INR): 19388965160.0

Sample:
 Depot ID  Effective KM  Calculated EV Energy (MWh)  Electricity Tariff (INR/kWh)  EV Electricity Cost (INR)
KSRTC-001        129270                   161.58750                             8                  1292700.0
KSRTC-001        129616                   162.02000                             8                  1296160.0
KSRTC-001        132462                   165.57750                             8                  1324620.0
KSRTC-001        140413                   175.51625                             8                  1404130.0
KSRTC-001        126289                   157.86125           

In [19]:
# Step 18: Annual EV energy requirement

annual_energy = (
    impact_df
    .groupby("Year")
    .agg(
        Total_Effective_KM=("Effective KM", "sum"),
        EV_Energy_MWh=("Calculated EV Energy (MWh)", "sum"),
        EV_Electricity_Cost_INR=("EV Electricity Cost (INR)", "sum")
    )
    .reset_index()
)

annual_energy["EV_Energy_GWh"] = (
    annual_energy["EV_Energy_MWh"] / 1000
)

print("Annual EV energy requirement")
print("-" * 70)

print(
    annual_energy[
        [
            "Year",
            "Total_Effective_KM",
            "EV_Energy_MWh",
            "EV_Energy_GWh",
            "EV_Electricity_Cost_INR"
        ]
    ].to_string(index=False)
)

print()

print(
    "Total EV energy requirement:",
    round(annual_energy["EV_Energy_GWh"].sum(), 2),
    "GWh"
)

print(
    "Total EV electricity cost:",
    round(annual_energy["EV_Electricity_Cost_INR"].sum(), 2),
    "INR"
)

Annual EV energy requirement
----------------------------------------------------------------------
 Year  Total_Effective_KM  EV_Energy_MWh  EV_Energy_GWh  EV_Electricity_Cost_INR
 2021           217354868   271693.58500     271.693585             2173548680.0
 2022           391403336   489254.17000     489.254170             3914033360.0
 2023           450041858   562552.32250     562.552323             4500418580.0
 2024           401406703   501758.37875     501.758379             4014067030.0
 2025           382855996   478569.99500     478.569995             3828559960.0
 2026            95833755   119792.19375     119.792194              958337550.0

Total EV energy requirement: 2423.62 GWh
Total EV electricity cost: 19388965160.0 INR


In [20]:
# Step 19: Depot-level impact summary

depot_impact = (
    impact_df
    .groupby(
        ["Depot ID", "Depot Name", "District"],
        as_index=False
    )
    .agg(
        Total_Effective_KM=("Effective KM", "sum"),
        Total_Diesel_Litres=("Estimated Diesel Litres", "sum"),
        Total_Diesel_CO2_Tonnes=("Estimated CO2 (Tonnes)", "sum"),
        Total_EV_Energy_MWh=("Calculated EV Energy (MWh)", "sum"),
        Total_EV_Electricity_Cost_INR=("EV Electricity Cost (INR)", "sum"),
        Total_Diesel_Cost_INR=("Diesel Cost (INR)", "sum"),
        Total_EV_Cost_Dataset_INR=(
            "EV Cost - Dataset Assumption (INR)",
            "sum"
        ),
        Total_Saving_Dataset_Model_INR=(
            "Saving - Dataset Cost Model (INR)",
            "sum"
        ),
        Total_Saving_Energy_Model_INR=(
            "Saving - Energy Cost Model (INR)",
            "sum"
        ),
        Average_Buses=("Buses Allocated", "mean"),
        Average_Schedules=("Schedules Allocated", "mean"),
        Average_Passengers=("Passengers", "mean")
    )
)

depot_impact["Potential_CO2_Reduction_Tonnes"] = (
    depot_impact["Total_Diesel_CO2_Tonnes"]
)

depot_impact["EV_Energy_GWh"] = (
    depot_impact["Total_EV_Energy_MWh"] / 1000
)

depot_impact["OPEX_Saving_Per_KM_INR"] = (
    depot_impact["Total_Saving_Dataset_Model_INR"]
    / depot_impact["Total_Effective_KM"]
)

depot_impact = depot_impact.sort_values(
    "Total_Saving_Dataset_Model_INR",
    ascending=False
).reset_index(drop=True)

print("Depot-level impact summary")
print("-" * 70)

print("Number of depots:", len(depot_impact))
print(
    "Total effective KM:",
    depot_impact["Total_Effective_KM"].sum()
)

print(
    "Total potential CO2 reduction (tonnes):",
    round(
        depot_impact["Potential_CO2_Reduction_Tonnes"].sum(),
        2
    )
)

print(
    "Total EV energy requirement (GWh):",
    round(
        depot_impact["EV_Energy_GWh"].sum(),
        2
    )
)

print(
    "Total dataset-model OPEX saving (INR):",
    round(
        depot_impact["Total_Saving_Dataset_Model_INR"].sum(),
        2
    )
)

print()

print("Top 10 depots by dataset-model OPEX saving:")
print(
    depot_impact[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Total_Effective_KM",
            "Total_Saving_Dataset_Model_INR",
            "Potential_CO2_Reduction_Tonnes",
            "EV_Energy_GWh"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Depot-level impact summary
----------------------------------------------------------------------
Number of depots: 92
Total effective KM: 1938896516
Total potential CO2 reduction (tonnes): 1273432.83
Total EV energy requirement (GWh): 2423.62
Total dataset-model OPEX saving (INR): 46533516384

Top 10 depots by dataset-model OPEX saving:
 Depot ID      Depot Name  District  Total_Effective_KM  Total_Saving_Dataset_Model_INR  Potential_CO2_Reduction_Tonnes  EV_Energy_GWh
KSRTC-024          KANNUR    Kannur            40430339                       970328136                    26553.929303      50.537924
KSRTC-037        KOTTAYAM  Kottayam            36188530                       868524720                    23767.984414      45.235662
KSRTC-032          KOLLAM    Kollam            35053641                       841287384                    23022.609455      43.817051
KSRTC-074 SULTHAN BATHERY   Wayanad            33010875                       792261000                    21680.956991 

In [21]:
# Step 20: Depot-level impact for the latest complete year (2025)

COMPLETE_YEAR = 2025

depot_annual_2025 = (
    impact_df[impact_df["Year"] == COMPLETE_YEAR]
    .groupby(
        ["Depot ID", "Depot Name", "District"],
        as_index=False
    )
    .agg(
        Effective_KM=("Effective KM", "sum"),
        Diesel_CO2_Tonnes=("Estimated CO2 (Tonnes)", "sum"),
        EV_Energy_MWh=("Calculated EV Energy (MWh)", "sum"),
        Diesel_Cost_INR=("Diesel Cost (INR)", "sum"),
        EV_Cost_Dataset_INR=(
            "EV Cost - Dataset Assumption (INR)",
            "sum"
        ),
        OPEX_Saving_INR=(
            "Saving - Dataset Cost Model (INR)",
            "sum"
        )
    )
)

depot_annual_2025["EV_Energy_GWh"] = (
    depot_annual_2025["EV_Energy_MWh"] / 1000
)

depot_annual_2025["OPEX_Saving_Crore"] = (
    depot_annual_2025["OPEX_Saving_INR"] / 1e7
)

depot_annual_2025["CO2_Reduction_Tonnes"] = (
    depot_annual_2025["Diesel_CO2_Tonnes"]
)

depot_annual_2025 = depot_annual_2025.sort_values(
    "OPEX_Saving_INR",
    ascending=False
).reset_index(drop=True)

print("Depot-level impact — 2025 complete year")
print("-" * 70)

print(
    "Number of depots:",
    len(depot_annual_2025)
)

print(
    "Total OPEX saving:",
    round(
        depot_annual_2025["OPEX_Saving_INR"].sum(),
        2
    ),
    "INR"
)

print(
    "Total OPEX saving:",
    round(
        depot_annual_2025["OPEX_Saving_Crore"].sum(),
        2
    ),
    "crore"
)

print()

print("Top 10 depots by 2025 OPEX saving:")

print(
    depot_annual_2025[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Effective_KM",
            "OPEX_Saving_INR",
            "OPEX_Saving_Crore",
            "CO2_Reduction_Tonnes",
            "EV_Energy_GWh"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Depot-level impact — 2025 complete year
----------------------------------------------------------------------
Number of depots: 92
Total OPEX saving: 9188543904 INR
Total OPEX saving: 918.85 crore

Top 10 depots by 2025 OPEX saving:
 Depot ID      Depot Name  District  Effective_KM  OPEX_Saving_INR  OPEX_Saving_Crore  CO2_Reduction_Tonnes  EV_Energy_GWh
KSRTC-024          KANNUR    Kannur       7837632        188103168          18.810317           5147.617635       9.797040
KSRTC-032          KOLLAM    Kollam       7465082        179161968          17.916197           4902.933405       9.331353
KSRTC-021        KALPETTA   Wayanad       6763262        162318288          16.231829           4441.990482       8.454077
KSRTC-044    MANANTHAVADY   Wayanad       6755460        162131040          16.213104           4436.866266       8.444325
KSRTC-037        KOTTAYAM  Kottayam       6725522        161412528          16.141253           4417.203519       8.406902
KSRTC-074 SULTHAN BATHERY   

In [22]:
# Step 21: 2025 OPEX saving sanity check

total_diesel_cost_2025 = depot_annual_2025["Diesel_Cost_INR"].sum()
total_ev_cost_2025 = depot_annual_2025["EV_Cost_Dataset_INR"].sum()
total_saving_2025 = depot_annual_2025["OPEX_Saving_INR"].sum()

saving_percentage = (
    total_saving_2025
    / total_diesel_cost_2025
    * 100
)

average_saving_per_depot = (
    total_saving_2025
    / len(depot_annual_2025)
)

print("2025 OPEX saving sanity check")
print("-" * 60)

print(
    "Total modeled diesel operating cost:",
    round(total_diesel_cost_2025 / 1e7, 2),
    "crore"
)

print(
    "Total modeled EV operating cost:",
    round(total_ev_cost_2025 / 1e7, 2),
    "crore"
)

print(
    "Total modeled OPEX saving:",
    round(total_saving_2025 / 1e7, 2),
    "crore"
)

print(
    "Modeled operating-cost reduction:",
    round(saving_percentage, 2),
    "%"
)

print(
    "Average modeled saving per depot:",
    round(average_saving_per_depot / 1e7, 2),
    "crore/year"
)

print()

print("Expected saving per kilometre:")
print(
    round(
        total_saving_2025
        / depot_annual_2025["Effective_KM"].sum(),
        2
    ),
    "INR/km"
)

2025 OPEX saving sanity check
------------------------------------------------------------
Total modeled diesel operating cost: 1952.57 crore
Total modeled EV operating cost: 1033.71 crore
Total modeled OPEX saving: 918.85 crore
Modeled operating-cost reduction: 47.06 %
Average modeled saving per depot: 9.99 crore/year

Expected saving per kilometre:
24.0 INR/km


In [25]:
# Step 22: Generate ML suitability predictions and combine with 2025 impact

import joblib

# Make sure Month is datetime
impact_df["Month"] = pd.to_datetime(
    impact_df["Month"],
    format="%Y-%m"
)

# Load trained deployment model
model_path = "../models/tejas_ev_suitability_rf.pkl"
final_model = joblib.load(model_path)

# Latest available month
latest_year = impact_df["Year"].max()

latest_df = impact_df[
    impact_df["Year"] == latest_year
].copy()

latest_month = latest_df["Month"].max()

# One latest-month record per depot
depot_baseline = (
    latest_df[
        latest_df["Month"] == latest_month
    ]
    [
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Buses Allocated",
            "Schedules Allocated",
            "Effective KM",
            "Passengers"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Model feature
depot_baseline["Passengers_per_Bus"] = (
    depot_baseline["Passengers"]
    / depot_baseline["Buses Allocated"]
)

# 2027 scenario assumptions
future_year = 2027

bus_growth = 0.10
schedule_growth = 0.05
km_growth = 0.08
passenger_growth = 0.07

future_df = depot_baseline.copy()

future_df["Buses Allocated"] = (
    future_df["Buses Allocated"]
    * (1 + bus_growth)
).round().astype(int)

future_df["Schedules Allocated"] = (
    future_df["Schedules Allocated"]
    * (1 + schedule_growth)
).round().astype(int)

future_df["Effective KM"] = (
    future_df["Effective KM"]
    * (1 + km_growth)
).round().astype(int)

future_df["Passengers"] = (
    future_df["Passengers"]
    * (1 + passenger_growth)
).round().astype(int)

future_df["Passengers_per_Bus"] = (
    future_df["Passengers"]
    / future_df["Buses Allocated"]
)

future_df["Year"] = future_year
future_df["Month_Number"] = latest_month.month

# Model features
model_features = [
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Year",
    "Month_Number",
    "Passengers_per_Bus"
]

future_X = future_df[model_features]

# Predict EV suitability
future_df["Predicted EV Suitability Score"] = (
    final_model.predict(future_X)
)

future_df["Predicted EV Suitability Score"] = (
    future_df["Predicted EV Suitability Score"]
    .clip(0, 1)
)

# ML scores
ml_scores = future_df[
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Predicted EV Suitability Score"
    ]
].copy()

# Merge with 2025 annual impact
depot_final_analysis = pd.merge(
    ml_scores,
    depot_annual_2025[
        [
            "Depot ID",
            "Effective_KM",
            "OPEX_Saving_INR",
            "OPEX_Saving_Crore",
            "CO2_Reduction_Tonnes",
            "EV_Energy_GWh"
        ]
    ],
    on="Depot ID",
    how="inner"
)

# ML ranking
depot_final_analysis["ML_Rank"] = (
    depot_final_analysis[
        "Predicted EV Suitability Score"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

depot_final_analysis = (
    depot_final_analysis
    .sort_values(
        "Predicted EV Suitability Score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("ML suitability + 2025 impact analysis")
print("-" * 70)

print("Number of depots:", len(depot_final_analysis))

print()

print("Top 10 depots by 2027 predicted EV suitability:")

print(
    depot_final_analysis[
        [
            "ML_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score",
            "OPEX_Saving_Crore",
            "CO2_Reduction_Tonnes",
            "EV_Energy_GWh"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

ML suitability + 2025 impact analysis
----------------------------------------------------------------------
Number of depots: 92

Top 10 depots by 2027 predicted EV suitability:
 ML_Rank  Depot ID      Depot Name  District  Predicted EV Suitability Score  OPEX_Saving_Crore  CO2_Reduction_Tonnes  EV_Energy_GWh
       1 KSRTC-032          KOLLAM    Kollam                        0.559076          17.916197           4902.933405       9.331353
       2 KSRTC-024          KANNUR    Kannur                        0.558387          18.810317           5147.617635       9.797040
       3 KSRTC-074 SULTHAN BATHERY   Wayanad                        0.533127          16.099613           4405.808347       8.385215
       4 KSRTC-021        KALPETTA   Wayanad                        0.532980          16.231829           4441.990482       8.454077
       5 KSRTC-044    MANANTHAVADY   Wayanad                        0.531508          16.213104           4436.866266       8.444325
       6 KSRTC-037     

In [26]:
# Step 23: Add terrain information to the final depot analysis

route_path = "../data/raw/ROUTE.csv"
route_df = pd.read_csv(route_path)

# Keep only the terrain information needed for depot-level analysis
terrain_info = route_df[
    [
        "Depot_ID",
        "Terrain_Class",
        "Terrain_Score"
    ]
].copy()

# Make sure Depot IDs have the same format
terrain_info["Depot_ID"] = terrain_info["Depot_ID"].astype(str)
depot_final_analysis["Depot ID"] = (
    depot_final_analysis["Depot ID"].astype(str)
)

# Merge terrain information
depot_final_analysis = pd.merge(
    depot_final_analysis,
    terrain_info,
    left_on="Depot ID",
    right_on="Depot_ID",
    how="left"
)

# Remove duplicate ID column
depot_final_analysis = depot_final_analysis.drop(
    columns=["Depot_ID"]
)

print("Terrain information merged")
print("-" * 60)

print(
    "Total depots:",
    len(depot_final_analysis)
)

print(
    "Missing terrain values:",
    depot_final_analysis[
        "Terrain_Class"
    ].isna().sum()
)

print()

print("Terrain distribution:")

print(
    depot_final_analysis[
        "Terrain_Class"
    ].value_counts()
)

print()

print("Sample:")
print(
    depot_final_analysis[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score",
            "Terrain_Class",
            "Terrain_Score"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Terrain information merged
------------------------------------------------------------
Total depots: 92
Missing terrain values: 0

Terrain distribution:
Terrain_Class
Flat/Rolling    57
Rolling         18
Flat             7
Steep            7
Hilly            3
Name: count, dtype: int64

Sample:
 Depot ID      Depot Name  District  Predicted EV Suitability Score Terrain_Class  Terrain_Score
KSRTC-032          KOLLAM    Kollam                        0.559076  Flat/Rolling           0.90
KSRTC-024          KANNUR    Kannur                        0.558387  Flat/Rolling           0.90
KSRTC-074 SULTHAN BATHERY   Wayanad                        0.533127         Hilly           0.40
KSRTC-021        KALPETTA   Wayanad                        0.532980         Hilly           0.40
KSRTC-044    MANANTHAVADY   Wayanad                        0.531508         Hilly           0.40
KSRTC-037        KOTTAYAM  Kottayam                        0.529478       Rolling           0.75
KSRTC-026       KASARGO

In [27]:
# Step 24: Calculate terrain-adjusted EV suitability

depot_final_analysis["Terrain_Adjusted_EV_Score"] = (
    depot_final_analysis["Predicted EV Suitability Score"]
    * depot_final_analysis["Terrain_Score"]
)

# Rank all depots
depot_final_analysis["Terrain_Adjusted_Rank"] = (
    depot_final_analysis["Terrain_Adjusted_EV_Score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

# Sort by adjusted suitability
depot_final_analysis = (
    depot_final_analysis
    .sort_values(
        "Terrain_Adjusted_EV_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Terrain-adjusted EV suitability analysis")
print("-" * 70)

print("Number of depots:", len(depot_final_analysis))

print()

print("Top 10 depots after terrain adjustment:")

print(
    depot_final_analysis[
        [
            "Terrain_Adjusted_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score",
            "Terrain_Class",
            "Terrain_Score",
            "Terrain_Adjusted_EV_Score"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print()

print("Score statistics:")

print(
    depot_final_analysis[
        "Terrain_Adjusted_EV_Score"
    ].describe()
)

Terrain-adjusted EV suitability analysis
----------------------------------------------------------------------
Number of depots: 92

Top 10 depots after terrain adjustment:
 Terrain_Adjusted_Rank  Depot ID     Depot Name  District  Predicted EV Suitability Score Terrain_Class  Terrain_Score  Terrain_Adjusted_EV_Score
                     1 KSRTC-032         KOLLAM    Kollam                        0.559076  Flat/Rolling            0.9                   0.503168
                     2 KSRTC-024         KANNUR    Kannur                        0.558387  Flat/Rolling            0.9                   0.502549
                     3 KSRTC-002      ALAPPUZHA Alappuzha                        0.490112          Flat            1.0                   0.490112
                     4 KSRTC-026      KASARGODE Kasaragod                        0.519530  Flat/Rolling            0.9                   0.467577
                     5 KSRTC-022      KANGANGAD Kasaragod                        0.516249  Flat/

In [28]:
# Step 25: Generate final EV deployment decision

def get_final_decision(row):

    terrain = row["Terrain_Class"]

    if terrain in ["Hilly", "Steep"]:
        return "Diesel Preferred"

    elif terrain == "Rolling":
        return "EV Conditional"

    elif terrain in ["Flat", "Flat/Rolling"]:
        if row["Terrain_Adjusted_EV_Score"] >= 0.40:
            return "EV Preferred"
        else:
            return "EV Conditional"

    return "EV Conditional"


depot_final_analysis["Final_EV_Decision"] = (
    depot_final_analysis.apply(
        get_final_decision,
        axis=1
    )
)

# Final decision counts
decision_counts = (
    depot_final_analysis[
        "Final_EV_Decision"
    ]
    .value_counts()
)

print("Final EV deployment decision")
print("-" * 70)

print(
    decision_counts.to_string()
)

print()

print("Decision percentages:")

print(
    (
        decision_counts
        / len(depot_final_analysis)
        * 100
    )
    .round(2)
    .to_string()
)

print()

print("Final decision distribution by terrain:")

print(
    pd.crosstab(
        depot_final_analysis["Terrain_Class"],
        depot_final_analysis["Final_EV_Decision"]
    )
)

print()

print("Top EV Preferred depots:")

print(
    depot_final_analysis[
        depot_final_analysis["Final_EV_Decision"]
        == "EV Preferred"
    ][
        [
            "Terrain_Adjusted_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score",
            "Terrain_Class",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Decision"
        ]
    ]
    .sort_values(
        "Terrain_Adjusted_EV_Score",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

Final EV deployment decision
----------------------------------------------------------------------
Final_EV_Decision
EV Conditional      48
EV Preferred        34
Diesel Preferred    10

Decision percentages:
Final_EV_Decision
EV Conditional      52.17
EV Preferred        36.96
Diesel Preferred    10.87

Final decision distribution by terrain:
Final_EV_Decision  Diesel Preferred  EV Conditional  EV Preferred
Terrain_Class                                                    
Flat                              0               0             7
Flat/Rolling                      0              30            27
Hilly                             3               0             0
Rolling                           0              18             0
Steep                             7               0             0

Top EV Preferred depots:
 Terrain_Adjusted_Rank  Depot ID           Depot Name           District  Predicted EV Suitability Score Terrain_Class  Terrain_Adjusted_EV_Score Final_EV_Decision
 

In [29]:
# Step 26: Final validation of depot-level impact analysis

print("FINAL IMPACT ANALYSIS VALIDATION")
print("=" * 70)

# 1. Row count
print("\n1. Row count")
print("Expected:", 92)
print("Actual:  ", len(depot_final_analysis))

# 2. Duplicate depots
print("\n2. Duplicate Depot IDs")
print(
    depot_final_analysis["Depot ID"].duplicated().sum()
)

# 3. Missing values
print("\n3. Missing values")
print(
    depot_final_analysis.isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

# 4. Score range
print("\n4. Score range")
print(
    "ML score:",
    depot_final_analysis[
        "Predicted EV Suitability Score"
    ].min(),
    "to",
    depot_final_analysis[
        "Predicted EV Suitability Score"
    ].max()
)

print(
    "Terrain-adjusted score:",
    depot_final_analysis[
        "Terrain_Adjusted_EV_Score"
    ].min(),
    "to",
    depot_final_analysis[
        "Terrain_Adjusted_EV_Score"
    ].max()
)

# 5. Terrain score validation
print("\n5. Terrain score values")
print(
    sorted(
        depot_final_analysis[
            "Terrain_Score"
        ].unique()
    )
)

# 6. Decision validation
print("\n6. Final decision counts")
print(
    depot_final_analysis[
        "Final_EV_Decision"
    ].value_counts()
)

# 7. Mathematical validation of terrain adjustment
print("\n7. Terrain-adjusted score formula validation")

calculated_adjusted = (
    depot_final_analysis[
        "Predicted EV Suitability Score"
    ]
    * depot_final_analysis["Terrain_Score"]
)

max_difference = (
    calculated_adjusted
    - depot_final_analysis[
        "Terrain_Adjusted_EV_Score"
    ]
).abs().max()

print("Maximum difference:", max_difference)

# 8. Ranking validation
print("\n8. Ranking validation")

expected_ranks = (
    depot_final_analysis[
        "Terrain_Adjusted_EV_Score"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

print(
    "Ranking matches:",
    expected_ranks.equals(
        depot_final_analysis[
            "Terrain_Adjusted_Rank"
        ]
    )
)

# 9. Impact values
print("\n9. 2025 impact totals")

print(
    "OPEX saving:",
    depot_final_analysis[
        "OPEX_Saving_INR"
    ].sum()
)

print(
    "CO2 reduction:",
    depot_final_analysis[
        "CO2_Reduction_Tonnes"
    ].sum()
)

print(
    "EV energy:",
    depot_final_analysis[
        "EV_Energy_GWh"
    ].sum()
)

print("\n" + "=" * 70)

if (
    len(depot_final_analysis) == 92
    and
    depot_final_analysis["Depot ID"].duplicated().sum() == 0
    and
    depot_final_analysis.isna().sum().sum() == 0
    and
    max_difference < 1e-10
    and
    expected_ranks.equals(
        depot_final_analysis[
            "Terrain_Adjusted_Rank"
        ]
    )
):
    print("FINAL VALIDATION: PASSED")
else:
    print("FINAL VALIDATION: CHECK REQUIRED")

FINAL IMPACT ANALYSIS VALIDATION

1. Row count
Expected: 92
Actual:   92

2. Duplicate Depot IDs
0

3. Missing values
Depot ID                          0
Depot Name                        0
District                          0
Predicted EV Suitability Score    0
Effective_KM                      0
OPEX_Saving_INR                   0
OPEX_Saving_Crore                 0
CO2_Reduction_Tonnes              0
EV_Energy_GWh                     0
ML_Rank                           0
dtype: int64

4. Score range
ML score: 0.28134950000000003 to 0.5590759999999995
Terrain-adjusted score: 0.06773249999999999 to 0.5031683999999995

5. Terrain score values
[np.float64(0.2), np.float64(0.4), np.float64(0.75), np.float64(0.9), np.float64(1.0)]

6. Final decision counts
Final_EV_Decision
EV Conditional      48
EV Preferred        34
Diesel Preferred    10
Name: count, dtype: int64

7. Terrain-adjusted score formula validation
Maximum difference: 0.0

8. Ranking validation
Ranking matches: True

9. 2025 

In [30]:
# Step 27: Export final depot-level analysis

output_path = "../outputs/TEJAS_FINAL_DEPOT_ANALYSIS.csv"

depot_final_analysis.to_csv(
    output_path,
    index=False
)

print("Final depot analysis exported successfully")
print("-" * 70)
print("File:", output_path)
print("Rows:", len(depot_final_analysis))
print("Columns:", len(depot_final_analysis.columns))

print("\nColumns:")
print(depot_final_analysis.columns.tolist())

Final depot analysis exported successfully
----------------------------------------------------------------------
File: ../outputs/TEJAS_FINAL_DEPOT_ANALYSIS.csv
Rows: 92
Columns: 15

Columns:
['Depot ID', 'Depot Name', 'District', 'Predicted EV Suitability Score', 'Effective_KM', 'OPEX_Saving_INR', 'OPEX_Saving_Crore', 'CO2_Reduction_Tonnes', 'EV_Energy_GWh', 'ML_Rank', 'Terrain_Class', 'Terrain_Score', 'Terrain_Adjusted_EV_Score', 'Terrain_Adjusted_Rank', 'Final_EV_Decision']
